# Lab 7.3 — Retrieval Tuning

Apply targeted fixes to the Lab 7.2 failure modes and quantify the improvement.

**Sweep dimensions:**
- `chunk_size` ∈ {512, 1024}
- `chunk_overlap` ∈ {50, 100}
- `top_k` ∈ {5, 10}
- `use_filters` ∈ {True, False}
- `use_reranker` ∈ {False, True (`bge-reranker-base`)}

Indexes are cached per `(chunk_size, chunk_overlap, embed_revision, manifest_hash)` so each unique embedding combo is built only once. Reranker is only applied to the best non-rerank config to keep wall-clock bounded.

Also: demonstrates the `AgentWorkflow` orchestration layer (`classify_query → retrieve → validate → synthesize → cite → log_failure_if_low_confidence`).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT.name != 'financial-report-analyst' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

## 1. Run the tuning sweep

Skip this cell and load the persisted `results/tuning.json` if you've already run `scripts/run_tuning.py` from the command line.

In [ ]:
from financial_analyst.tune import run_full_sweep_with_rerank, rows_to_dataframe, write_tuning_outputs
rows = run_full_sweep_with_rerank()
json_path, md_path = write_tuning_outputs(rows)
print(f'Wrote {json_path} and {md_path}')

## 2. Inspect the comparison table

In [ ]:
import pandas as pd
df = rows_to_dataframe(rows).sort_values('strict_mrr', ascending=False)
df.head(20)

## 3. Pre/post comparison vs Lab 7.2 baseline

In [ ]:
BASELINE = df[(df.chunk_size == 512) & (df.overlap == 50) & (df.top_k == 5) & (df.filters) & (~df.rerank)].iloc[0]
BEST = df.iloc[0]
pd.DataFrame([
    {'config': 'baseline (Lab 7.2)', **BASELINE.to_dict()},
    {'config': 'best (Lab 7.3)',    **BEST.to_dict()},
])

## 4. Demonstrate the AgentWorkflow

Six-step pipeline: `classify_query → retrieve → validate_sources → synthesize_answer → produce_citations → log_failure_if_low_confidence`. The agent appends to `results/agent_failures.jsonl` whenever the top similarity score is below 0.55.

In [ ]:
from financial_analyst.workflow import run_agent_query

for q in [
    "How did NVIDIA's data center revenue change from FY2023 to FY2024?",
    "What is the company's plan for quantum computing?",  # intentionally OOD
]:
    print('=' * 80)
    print('Q:', q)
    print()
    print(run_agent_query(q).render()[:1500])
    print()

## 5. Sample low-confidence log

Check `results/agent_failures.jsonl` for the appended record.

In [ ]:
import json, pathlib
p = pathlib.Path('../results/agent_failures.jsonl')
if p.exists():
    for line in p.read_text(encoding='utf-8').splitlines()[-5:]:
        print(json.dumps(json.loads(line), indent=2))
else:
    print('No failures logged yet.')